In [1]:
from pymongo import MongoClient
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI not found in .env")

In [2]:
client = MongoClient(MONGO_URI)
src_db = client["hr"]
dst_db = client["hr-cleaned"]


## Cleaning Base Report

hr.base_report -> hr-cleaned.base_report

In [3]:
src_col = src_db["base_report"]
dst_col = dst_db["base_report"]

In [ ]:
fields = [
    "_id", 
    "assignment status type", 
    "date_of_birth", 
    "date_of_joining",
    "department", 
    "designation", 
    "employee code", 
    "first name", 
    "grade",
    "grade level",
    "last name", 
    "location", 
    "m/f",
    "office", 
    "region",
    "state", 
    "sub-dept", 
    "circle", 
    "date_of_leaving",
    "date_of_resignation",
    "last year rating", 
    "manager code", 
    "primary email", 
    "reason for resignation",
    "reporting to", 
    "role"
]

key_renames = {
    "m/f": "gender",
    "manager code": "manager employee code",
    "primary email": "email"
}

numeric_fields = {
    "last year rating",
    "employee code",
    "manager employee code"    
}

string_or_date_fields = set(numeric_fields) - { "_id" } - {
    "last year rating",
    "employee code",
    "manager code"              # original key
}

In [14]:
# def normalize_doc(raw_doc,FIELDS,KEY_RENAMES,NUMERIC_FIELDS):
#     cleaned = {}
#     for field in FIELDS:

#         old_key = field
#         new_key = KEY_RENAMES.get(field, field)

#         # Extracting original value
#         value = raw_doc.get(old_key)

#         # Determine missing values
#         if value is None:
#             if new_key in NUMERIC_FIELDS:
#                 cleaned[new_key] = None
#             else:
#                 cleaned[new_key] = "NA"
#             continue

#         # Field exists → Normalize it
#         if new_key in NUMERIC_FIELDS:
#             # Try to cast to integer, else None
#             try:
#                 cleaned[new_key] = int(value)
#             except Exception:
#                 print(f"Warning: Could not convert {value} to int for key {new_key}.")
#                 cleaned[new_key] = value
#         else:
#             # Convert to cleaned string
#             cleaned[new_key] = str(value).strip() if str(value).strip() else "NA"

#     # Ensure _id is preserved
#     cleaned["_id"] = raw_doc["_id"]

#     return cleaned
def normalize_doc_generic(raw_doc, fields, key_renames, numeric_fields):
    cleaned = {}

    for field in fields:
        old_key = field
        new_key = key_renames.get(field, field)

        value = raw_doc.get(old_key)

        if value is None:
            cleaned[new_key] = None if new_key in numeric_fields else "NA"
            continue

        if new_key in numeric_fields:
            try:
                cleaned[new_key] = int(value)
            except:
                cleaned[new_key] = None
        else:
            cleaned[new_key] = str(value).strip() if str(value).strip() else "NA"

    cleaned["_id"] = raw_doc["_id"]
    return cleaned


In [10]:
batch = []
for doc in src_col.find({}):
    cleaned = normalize_doc(doc,fields,key_renames,numeric_fields)
    batch.append(cleaned)

    if len(batch) == 1000:
        dst_col.insert_many(batch)
        batch = []

if batch:
    dst_col.insert_many(batch)

print("Cleaning complete. Documents inserted:", dst_col.count_documents({}))

Cleaning complete. Documents inserted: 7859


## Cleaning Goal Setting Status
hr.manager employee code -> hr-cleaned.manager employee code

In [12]:
src_col = src_db["goal_setting_status"]
dst_col = dst_db["goal_setting_status"]

In [15]:
fields = [
    "_id",
    "Date of Joining",
    "department",
    "designation",
    "Employee name",
    "manager number",
    "office location",
    "parent grade",
    "person number",
    "region",
    "reporting to",
    "reviewer name",
    "reviewer number",
    "status",
    "sub department",
    "sub grade",
    "work location"
]

key_renames = {
    "Employee name": "employee name",
    "Date of Joining": "date_of_joining",
    "manager number": "manager employee code",
    "office location": "office",
    "work location": "location",
    "reviewer number": "reviewer employee code",
    "person number": "employee code",
    "parent grade": "grade",
    "sub grade": "grade level",
    "sub department": "sub-dept",
}

numeric_fields = {
    "reviewer employee code",
    "manager employee code",
    "employee code"
}

In [16]:
batch = []
for doc in src_col.find({}):
    batch.append(normalize_doc_generic(doc,fields,key_renames,numeric_fields))

    if len(batch) == 1000:
        dst_col.insert_many(batch)
        batch = []

if batch:
    dst_col.insert_many(batch)

print("goal_setting_status cleaned. Total:", dst_col.count_documents({}))

goal_setting_status cleaned. Total: 1292


## Cleaning Performance Goal Report 2025 26

hr.performance_goal_report_2025_2026 -> hr-cleaned.performance_goal_report_2025_2026

In [17]:
src_col = src_db["performance_goal_report_2025_2026"]
dst_col = dst_db["performance_goal_report_2025_2026"]

In [18]:
FIELDS = [
    "_id",
    "department",
    "description",
    "goal name",
    "goal plan name",
    "grade",
    "name",
    "person number",
    "region",
    "review period name",
    "sub department",
    "sub grade",
    "weight"
]

KEY_RENAMES = {
    "name": "employee name",
    "person number": "employee code",
    "sub department": "sub-dept",
    "sub grade": "grade level",
}

NUMERIC_FIELDS = {
    "employee code",
    "weight"
}

In [20]:
batch = []
for doc in src_col.find({}):
    cleaned = normalize_doc_generic(
        raw_doc=doc,
        fields=FIELDS,
        key_renames=KEY_RENAMES,
        numeric_fields=NUMERIC_FIELDS
    )
    batch.append(cleaned)

    if len(batch) >= 1000:
        dst_col.insert_many(batch)
        batch = []

if batch:
    dst_col.insert_many(batch)

print("performance_goal_report_2025_2026 cleaned. Total:",dst_col.count_documents({}))

performance_goal_report_2025_2026 cleaned. Total: 5047


## Cleaning Permormance Rating Report Year 2025 26
hr.permormance_rating_report_year_2025_2026 -> hr-cleaned.permormance_rating_report_year_2025_2026

In [22]:
src_col = src_db["permormance_rating_report_year_2025_2026"]
dst_col = dst_db["permormance_rating_report_year_2025_2026"]


In [23]:
FIELDS = [
    "_id",
    "business unit name",
    "department",
    "employee e mail address",
    "employee name",
    "employee person number",
    "final status",
    "grade",
    "job",
    "location name",
    "manager e mail address",
    "manager name",
    "manager person number",
    "manager rating",
    "parent department",
    "parent grade",
    "performance document name",
    "self rating"
]

KEY_RENAMES = {
    "employee e mail address": "email",
    "employee person number": "employee code",
    "business unit name": "region",
    "job": "designation",
    "parent grade": "grade",
    "grade": "grade level",
    "parent department": "department",
    "department": "sub-dept",
    "manager person number": "manager employee code",
    "manager e mail address": "manager email",
    "location name": "office"
}

NUMERIC_FIELDS = {
    "self rating",
    "manager rating",
    "employee code",
    "manager employee code"
}


In [ ]:
batch = []
for doc in src_col.find({}):
    cleaned = normalize_doc_generic(
        raw_doc=doc,
        fields=FIELDS,
        key_renames=KEY_RENAMES,
        numeric_fields=NUMERIC_FIELDS
    )
    batch.append(cleaned)

    if len(batch) >= 1000:
        dst_col.insert_many(batch)
        batch = []

if batch:
    dst_col.insert_many(batch)

print("permormance_rating_report_year_2025_2026 cleaned. Total:",dst_col.count_documents({}))


permormance_rating_report_year_2025_2026 cleaned. Total: 897
